# 12a WILDTRACK MVDeTr Training
Train MVDeTr on WILDTRACK, export ground-plane detections, convert them into repo-native tracklets and trajectories, and run ground-plane evaluation.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
PROJECT = WORK / "gp"
MVDETR = Path("/kaggle/working/MVDeTr")
DATA_HOME = Path.home() / "Data"
WT_HOME = DATA_HOME / "Wildtrack"
EXPORT_ROOT = PROJECT / "data" / "outputs" / "wildtrack_mvdetr"

REPO_URL = "https://github.com/MRKDaGods/gp.git"
if not PROJECT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy<2", "scipy", "scikit-learn", "opencv-python-headless",
    "kornia", "motmetrics", "matplotlib", "pillow", "lxml"
])
print(f"Repo ready at {PROJECT}")
print(f"MVDeTr path reserved at {MVDETR}")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MVDETR = Path("/kaggle/working/MVDeTr")
MVDETR_URL = "https://github.com/hou-yz/MVDeTr.git"
if not MVDETR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", MVDETR_URL, str(MVDETR)])
else:
    subprocess.check_call(["git", "-C", str(MVDETR), "pull", "--ff-only"])

os.chdir(MVDETR)
req = MVDETR / "requirements.txt"
if req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ninja", "packaging"])

def replace_once(path: Path, old: str, new: str) -> None:
    text = path.read_text(encoding="utf-8")
    if new in text:
        return
    if old not in text:
        raise RuntimeError(f"Could not patch {path}: expected snippet not found")
    path.write_text(text.replace(old, new, 1), encoding="utf-8")

def patch_mvdetr_python_fallback(repo_root: Path) -> None:
    func_path = repo_root / "multiview_detector" / "models" / "ops" / "functions" / "ms_deform_attn_func.py"
    module_path = repo_root / "multiview_detector" / "models" / "ops" / "modules" / "ms_deform_attn.py"
    replace_once(
        func_path,
        "import MultiScaleDeformableAttention as MSDA"
        ,
        (
            "try:\
"
            "    import MultiScaleDeformableAttention as MSDA\
"
            "except ImportError:\
"
            "    MSDA = None\
\
"
            "MSDA_AVAILABLE = MSDA is not None"
        ),
    )
    replace_once(
        module_path,
        "from ..functions import MSDeformAttnFunction"
        ,
        (
            "from ..functions.ms_deform_attn_func import (\
"
            "    MSDA_AVAILABLE,\
"
            "    MSDeformAttnFunction,\
"
            "    ms_deform_attn_core_pytorch,\
"
            ")"
        ),
    )
    replace_once(
        module_path,
        (
            "        output = MSDeformAttnFunction.apply(value, input_spatial_shapes, input_level_start_index,\
"
            "                                            sampling_locations, attention_weights, self.im2col_step)\
"
            "        output = self.output_proj(output)\
"
            "        return output"
        ),
        (
            "        if value.is_cuda and MSDA_AVAILABLE:\
"
            "            try:\
"
            "                output = MSDeformAttnFunction.apply(\
"
            "                    value,\
"
            "                    input_spatial_shapes,\
"
            "                    input_level_start_index,\
"
            "                    sampling_locations,\
"
            "                    attention_weights,\
"
            "                    self.im2col_step,\
"
            "                )\
"
            "            except RuntimeError as exc:\
"
            "                if 'Not compiled with GPU support' not in str(exc):\
"
            "                    raise\
"
            '                print(f"Falling back to PyTorch deformable attention: {exc}")\n'
            "                output = ms_deform_attn_core_pytorch(\
"
            "                    value,\
"
            "                    input_spatial_shapes,\
"
            "                    sampling_locations,\
"
            "                    attention_weights,\
"
            "                )\
"
            "        else:\
"
            "            output = ms_deform_attn_core_pytorch(\
"
            "                value,\
"
            "                input_spatial_shapes,\
"
            "                sampling_locations,\
"
            "                attention_weights,\
"
            "            )\
"
            "        output = self.output_proj(output)\
"
            "        return output"
        ),
    )
    print("Patched MVDeTr deformable attention fallback")

patch_mvdetr_python_fallback(MVDETR)
ops_dir = MVDETR / "multiview_detector" / "models" / "ops"
mask_script = ops_dir / "mask.sh"
make_script = ops_dir / "make.sh"
build_script = mask_script if mask_script.exists() else make_script
if not build_script.exists():
    raise FileNotFoundError(f"No MVDeTr build script found under {ops_dir}")

build_env = os.environ.copy()
build_env.setdefault("MAX_JOBS", "2")
build_env.setdefault("TORCH_CUDA_ARCH_LIST", "6.0;7.5")
try:
    subprocess.check_call(["bash", str(build_script)], cwd=ops_dir, env=build_env)
    print("Built MVDeTr CUDA extension")
except subprocess.CalledProcessError as exc:
    print(f"MVDeTr CUDA extension build failed ({exc}); using PyTorch fallback.")

print(f"MVDeTr ready at {MVDETR}")

In [ ]:
import re
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working")

def _find_wildtrack_mount():
    candidates = [
        Path("/kaggle/input/large-scale-multicamera-detection-dataset"),
        Path("/kaggle/input/datasets/aryashah2k/large-scale-multicamera-detection-dataset"),
        Path("/kaggle/input/wildtrack"),
        Path("/kaggle/input/datasets/aryashah2k/wildtrack"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def _best_effort_official_download():
    page = WORK / "wildtrack_official.html"
    subprocess.run(["wget", "-q", "-O", str(page), "https://www.epfl.ch/labs/cvlab/data/data-wildtrack/"], check=False)
    if not page.exists():
        return None
    html = page.read_text(encoding="utf-8", errors="ignore")
    urls = []
    for match in re.findall(r"href=[\"']([^\"']+)", html):
        if any(match.endswith(ext) for ext in (".zip", ".tar", ".tar.gz", ".tgz")):
            if match.startswith("http"):
                urls.append(match)
            else:
                urls.append("https://www.epfl.ch" + match if match.startswith("/") else match)
    download_dir = WORK / "wildtrack_official_download"
    download_dir.mkdir(exist_ok=True)
    for url in urls:
        target = download_dir / Path(url).name
        result = subprocess.run(["wget", "-q", "-O", str(target), url], check=False)
        if result.returncode == 0 and target.exists() and target.stat().st_size > 0:
            return download_dir
    return None

WILDTRACK_INPUT = _find_wildtrack_mount()
if WILDTRACK_INPUT is None:
    downloaded = _best_effort_official_download()
    if downloaded is None:
        raise FileNotFoundError("No mounted WILDTRACK dataset found and official download fallback did not resolve a usable archive. Attach a WILDTRACK Kaggle dataset.")
    WILDTRACK_INPUT = downloaded

print(f"WILDTRACK input root: {WILDTRACK_INPUT}")

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
DATA_HOME = Path.home() / "Data"
WT_HOME = DATA_HOME / "Wildtrack"

if "WILDTRACK_INPUT" not in globals():
    raise RuntimeError("Run the previous dataset-discovery cell first so WILDTRACK_INPUT is defined.")

def _find_named_dir(root: Path, name: str):
    for candidate in [root / name, *root.rglob(name)]:
        if candidate.exists():
            return candidate
    return None

img_root = _find_named_dir(WILDTRACK_INPUT, "Image_subsets")
ann_root = _find_named_dir(WILDTRACK_INPUT, "annotations_positions")
cal_root = _find_named_dir(WILDTRACK_INPUT, "calibrations")
if img_root is None or ann_root is None or cal_root is None:
    raise FileNotFoundError("WILDTRACK dataset is missing Image_subsets, annotations_positions, or calibrations")

WT_HOME.parent.mkdir(parents=True, exist_ok=True)
if WT_HOME.exists() or WT_HOME.is_symlink():
    if WT_HOME.is_symlink() or WT_HOME.is_file():
        WT_HOME.unlink()
    else:
        shutil.rmtree(WT_HOME)
WT_HOME.mkdir(parents=True, exist_ok=True)
for name, src in {"Image_subsets": img_root, "annotations_positions": ann_root, "calibrations": cal_root}.items():
    dst = WT_HOME / name
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    try:
        dst.symlink_to(src, target_is_directory=True)
    except OSError:
        shutil.copytree(src, dst)

local_wt = PROJECT / "data" / "raw" / "wildtrack"
local_wt.mkdir(parents=True, exist_ok=True)
for name, src in {"annotations_positions": ann_root, "calibrations": cal_root}.items():
    dst = local_wt / name
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    try:
        dst.symlink_to(src, target_is_directory=True)
    except OSError:
        shutil.copytree(src, dst)

subprocess.check_call([
    sys.executable, "scripts/prepare_dataset.py",
    "--dataset", "wildtrack",
    "--root", "data/raw/wildtrack",
])
print(f"Prepared upstream root: {WT_HOME}")
print(f"Prepared repo root: {local_wt}")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MVDETR = Path("/kaggle/working/MVDeTr")
if not MVDETR.exists():
    raise FileNotFoundError(f"MVDeTr repo not found at {MVDETR}; run the clone/build cell first.")

os.chdir(MVDETR)
EPOCHS = 10
TRAIN_ARGS = [
    sys.executable, "main.py",
    "-d", "wildtrack",
    "--arch", "resnet18",
    "--world_feat", "deform_trans",
    "--use_mse", "false",
    "--epochs", str(EPOCHS),
    "--batch_size", "1",
    "--num_workers", "2",
    "--lr", "5e-4",
    "--world_reduce", "4",
    "--world_kernel_size", "10",
    "--img_reduce", "12",
    "--img_kernel_size", "10",
    "--dropout", "0.0",
    "--dropcam", "0.0",
]
print('Running:', ' '.join(TRAIN_ARGS))
subprocess.check_call(TRAIN_ARGS)

log_root = MVDETR / "logs" / "wildtrack"
run_dirs = sorted([p for p in log_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not run_dirs:
    raise FileNotFoundError(f"No MVDeTr run directories found under {log_root}")
LATEST_RUN = run_dirs[-1]
TEST_TXT = LATEST_RUN / "test.txt"
CKPT = LATEST_RUN / "MultiviewDetector.pth"
if not TEST_TXT.exists():
    raise FileNotFoundError(f"Missing MVDeTr detections: {TEST_TXT}")
if not CKPT.exists():
    raise FileNotFoundError(f"Missing MVDeTr checkpoint: {CKPT}")
print(f"Latest run: {LATEST_RUN.name}")
print(f"Checkpoint: {CKPT}")
print(f"Detections: {TEST_TXT}")

In [ ]:
import os
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
EXPORT_ROOT = PROJECT / "data" / "outputs" / "wildtrack_mvdetr"
MVDETR = Path("/kaggle/working/MVDeTr")
if "LATEST_RUN" not in globals() or "TEST_TXT" not in globals():
    log_root = MVDETR / "logs" / "wildtrack"
    run_dirs = sorted([p for p in log_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise FileNotFoundError(f"No MVDeTr run directories found under {log_root}")
    LATEST_RUN = run_dirs[-1]
    TEST_TXT = LATEST_RUN / "test.txt"
if not TEST_TXT.exists():
    raise FileNotFoundError(f"Missing MVDeTr detections: {TEST_TXT}")

os.chdir(PROJECT)
from src.stage_wildtrack_mvdetr.pipeline import run_stage_wildtrack_mvdetr

converted_dir = EXPORT_ROOT / LATEST_RUN.name
tracklets_by_camera, trajectories = run_stage_wildtrack_mvdetr(
    detections_path=TEST_TXT,
    calibration_dir=PROJECT / "data" / "raw" / "wildtrack" / "calibrations",
    output_dir=converted_dir,
    fps=2.0,
    max_match_distance_cm=75.0,
    max_missed_frames=5,
    min_track_length=2,
)
print(f"Converted output dir: {converted_dir}")
print(f"Projected tracklets: {sum(len(v) for v in tracklets_by_camera.values())}")
print(f"Global trajectories: {len(trajectories)}")

In [ ]:
import json
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
if "trajectories" not in globals():
    raise RuntimeError("Run the previous conversion cell first so trajectories are available for evaluation.")
if "LATEST_RUN" not in globals():
    log_root = Path("/kaggle/working/MVDeTr") / "logs" / "wildtrack"
    run_dirs = sorted([p for p in log_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise FileNotFoundError(f"No MVDeTr run directories found under {log_root}")
    LATEST_RUN = run_dirs[-1]
if "converted_dir" not in globals():
    converted_dir = PROJECT / "data" / "outputs" / "wildtrack_mvdetr" / LATEST_RUN.name

from src.stage5_evaluation.ground_plane_eval import evaluate_wildtrack_ground_plane

result = evaluate_wildtrack_ground_plane(
    trajectories=trajectories,
    annotations_dir=PROJECT / "data" / "raw" / "wildtrack" / "annotations_positions",
    calibrations_dir=PROJECT / "data" / "raw" / "wildtrack" / "calibrations",
    conf_threshold=0.25,
    match_threshold_cm=50.0,
    nms_radius_cm=50.0,
)
summary = {
    "run_name": LATEST_RUN.name,
    "moda": result.mota,
    "idf1": result.idf1,
    "id_switches": result.id_switches,
    "details": result.details,
}
summary_path = converted_dir / "ground_plane_eval_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print(f"Saved summary to {summary_path}")